# Project 1: emotion classification on ISEAR

Comparing 3 representations × 3 architectures with cross-validation and a Bayesian ROPE analysis, reusing the tutorial code in `part1/`.

| Section | Who |
| --- | --- |
| 1. Data, 2. Folds, 3. Tokeniser | Britt |
| 4. Representations | Zeno |
| 5. Architectures | Armani |
| 6. Experiments, 7. Bayesian comparison, 8. Error analysis | Together |

Run `python explore_data.py` once first: it creates `data/isear_clean.csv`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import part1
from part1.clslib import NamedClasses

results_folder = Path("results/project1")
results_folder.mkdir(parents=True, exist_ok=True)

## 1. Data
Cleaned ISEAR (see `explore_data.py`): 7,516 raw texts, 84 removed (10 placeholders, 21 same text with conflicting labels, 53 duplicates), 7,432 left. The classes are balanced, so accuracy is a fair metric (majority baseline 14.5%).

In [3]:
LABELS = ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"]
outcome_space = NamedClasses(LABELS)

df = pd.read_csv("data/isear_clean.csv")
pairs = list(zip(df["text"], df["label_id"]))  # (text, label) pairs, as in the tutorial
print(len(pairs), "texts")
df["label"].value_counts()

7432 texts


label
joy        1079
anger      1077
fear       1070
disgust    1060
shame      1053
sadness    1050
guilt      1043
Name: count, dtype: int64

## 2. Cross-validation folds
5 folds, repeated with 3 seeds (15 accuracies per system), using the tutorial's `cross_validation_split`. Every system uses these same folds, so the scores are paired for the Bayesian comparison.

In [4]:
seeds = [0, 3, 5]
num_folds = 5
cv_sets = [part1.experiment.cross_validation_split(pairs, k=num_folds, seed=seed) for seed in seeds]

# check: are the classes spread evenly over the folds?
for fold, split in cv_sets[0].items():
    counts = pd.Series([label for _, label in split["validation"]]).value_counts().sort_index()
    print(f"fold {fold}: {len(split['validation'])} texts, per class {counts.tolist()}")

fold 0: 1487 texts, per class [231, 206, 186, 239, 219, 210, 196]
fold 1: 1487 texts, per class [198, 217, 227, 199, 215, 208, 223]
fold 2: 1486 texts, per class [237, 186, 228, 202, 221, 219, 193]
fold 3: 1486 texts, per class [207, 221, 219, 207, 206, 216, 210]
fold 4: 1486 texts, per class [206, 240, 217, 203, 199, 200, 221]


## 3. Tokeniser
BPE tokeniser trained with the tutorial's `train_tokenizer` on the ISEAR texts themselves (~165k words is enough for a small vocabulary). It uses no labels, so training on all texts does not leak label information (the tutorial does the same).

To choose the vocabulary size, we compare how finely each size splits words: a good size keeps frequent words, such as emotion words, whole without making the vocabulary huge.

In [5]:
texts = [text for text, _ in pairs]
n_words = sum(len(t.split()) for t in texts)
example = "I felt ashamed when I shouted at my girlfriend."

for size in [500, 1000, 2000, 5000]:
    tok = part1.experiment.train_tokenizer(texts, size, min_frequency=2)
    n_tokens = sum(len(tok.encode(t).ids) for t in texts)
    print(f"{size:>5}: {n_tokens / n_words:.2f} tokens per word | {tok.encode(example).tokens}")




  500: 2.10 tokens per word | ['▁I', '▁felt', '▁as', 'ham', 'ed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girl', 'friend', '.']



 1000: 1.74 tokens per word | ['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girlfriend', '.']



 2000: 1.50 tokens per word | ['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girlfriend.']



 5000: 1.28 tokens per word | ['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁shouted', '▁at', '▁my', '▁girlfriend.']


In [6]:
vocab_size = 1000  # same as tutorial (500 is too small and with 2000 puctuation gets glued on the words)
tokenizer = part1.experiment.train_tokenizer(texts, vocab_size, min_frequency=2)
tokenizer.save(f"{results_folder}/bpe_vs{vocab_size}.json")
print(tokenizer.encode(example).tokens)




['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girlfriend', '.']


## 4. Representations (Zeno)
- token indicators (tutorial)
- bag-of-words (tutorial exercise)
- one extra of our choice

In [ ]:
from collections import Counter
from tokenizers import Tokenizer

class Encoder:

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")


class TokenIndicator(Encoder):
    """h_v(x) = 1 if BPE token v occurs in x, else 0 (tutorial)."""

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
        
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(set(self.tokenizer.encode(x).ids))

        # Set the 0 vector to 1 in every word that appears
        vector = np.zeros(len(self))
        vector[unique_token_ids] = 1
        return vector


class BagOfWords(Encoder):
    """h_v(x) = number of times BPE token v occurs in x (tutorial exercise)."""

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
        self.dim = tokenizer.get_vocab_size()

    def __len__(self) -> int:
        return self.dim

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(self.dim)
        # np.add.at accumulates repeated ids; vector[ids] += 1 would count each id only once
        np.add.at(vector, self.tokenizer.encode(x).ids, 1)
        return vector


class BagOfUnigramsAndBigrams(Encoder):
    """
    Counts of BPE tokens plus counts of adjacent token pairs (the tutorial's 'bag of bigrams').
    Only bigrams seen at least `min_count` times in `texts` get a dimension, otherwise D ~ V^2.
    """

    def __init__(self, tokenizer: Tokenizer, texts: list, min_count: int = 2):
        self.tokenizer = tokenizer
        self.num_unigrams = tokenizer.get_vocab_size()
        bigram_counts = Counter(
            pair
            for text in texts
            for pair in self._bigrams(tokenizer.encode(text).ids)
        )
        kept = sorted(pair for pair, count in bigram_counts.items() if count >= min_count)
        # bigram dimensions come after the V unigram dimensions
        self.bigram_index = {pair: self.num_unigrams + i for i, pair in enumerate(kept)}
        self.dim = self.num_unigrams + len(self.bigram_index)

    @staticmethod
    def _bigrams(ids):
        return zip(ids, ids[1:])

    def __len__(self) -> int:
        return self.dim

    def __call__(self, x: str) -> np.ndarray:
            ids = self.tokenizer.encode(x).ids
            vector = np.zeros(self.dim)
            np.add.at(vector, ids, 1)
            bigram_dims = [self.bigram_index[p] for p in self._bigrams(ids) if p in self.bigram_index]
            np.add.at(vector, bigram_dims, 1)  # unseen bigrams are simply dropped
            return vector
    
    """FOR SECTION 8!
    def coords(self):
        Readable axis labels, e.g. for inspecting learned weights in the error analysis.

        id_to_token = {i: t for t, i in self.tokenizer.get_vocab().items()}
        labels = [id_to_token[i] for i in range(self.num_unigrams)]
        labels += [id_to_token[a] + " " + id_to_token[b] for (a, b) in self.bigram_index]
        return labels
    """
    

## 5. Architectures (Armani)
Log-linear and 1-hidden-layer MLP, plus one extra architecture

In [ ]:
# Log Linear

# f(x, y) -> feature function and returns vectors (in R^D)
# Y -> set of possible labels (7)
# D -> training data (x_i, y_i)
def score(w, x):
    ...
    # return dot(w, f(x, y))

def probabilities(w, x, Y): # turn scores into probabilities
    ...
    # s[y] = score(w, x, y) for every y in Y
    # m = max(s) find the largest score
    # Z = sum over y of exp(s[y]-m) -> softmax denominator. shift by m changes nothing mathematically but avoids bugs
    # return exp(s[y]-m)/Z for every y in Y (softmax probability vector that sums to 1)

def train(D, lr, reg_strength, num_epochs):
    ...
    # w = zeros(D) -> init all weights to zero
    # repeat the next part for every epoch in epochs
    # grad = zeros(D) -> full batch GD, can decide to change later tho
    # for every x, and y (label) in D:
    # p = probabilities(w, x) -> model's curr belief abt each label for datapoint x
    # grad += f(x, y)
    # for y in Y (3rd order loop)
    # grad -= p[y] * f(x, y)
    # grad -= reg_strength * w -> L2 penalty
    # w = w + lr * grad -> step, move w uphill on llh # maxxing the llh, but can also min the -llh
    # return w (learned weights)

def predict(w, x):
    ...
    # return argmax over y in Y of score(w, x, y) (most likely label given x)


In [2]:
# MLP

def softmax(scores):
    ...
    # m = max(scores)
    # e = exp(s-m)
    # return e/sum(e)

def forward(params, x):
    ...
    # W1, b1, W2, b2 = params -> W1: Hxd, b1:H, W2: KxH, b2: K
    # z = (W1 times x) + b1 -> Wx+b
    # h = relu(z) -> nonlinearity
    # s = (W2 times h) + b2
    # p = softmax(s)
    # return z, h, p -> z, h needed for bwd pass

def train(D, d, H, K, lr, reg_strength, num_epochs):
    ...
    # W1 = random_normal(H,d) * 0.01 -> small random, but NOT 0
    # b1 = zeros(H) -> biases are OK to start at 0
    # W2 = random_normal(K, H) * 0.01
    # b2 = np.zeros(K)
    # for epoch in num_epochs:
    # gW1, gb1, gW2, gb2 = zeros
    # for (x,y) in D (dataset):
    # z, h, p = forward(params, x)
    # now the backward pass (chain rule)
    # ds = p - onehot(y, K) -> torch.nn.functional.one_hot
    # gW2 += outer(ds, h)
    # gb2 += ds
    # dh W2.T times ds -> send the error back though to the hidden layer
    # dz = dh * (z > 0) -> relu grad passes through only where z > 0
    # gW1 += outer(dz, x)
    # gb1 += dz
    # gW1 += reg_strength*W1
    # gW2 += reg_strength*W2
    # W1 -= lr * gW1
    # b1 -= lr * gb1
    # W2 -= lr * gW2
    # b2 -= lr * gb2
    # return W1, b1, W2, b2

def predict(params, x):
    ...
    # _, _, p = forward(params, x)
    # return argmax(p) -> or argmax(s), but p better

## 6. Experiments (together)


## 7. Bayesian comparison with a ROPE (together)
Use tutorial 

## 8. Error analysis (together)
per-class F1 and a confusion matrix